# Car Damage Detection — YOLOv8 Object Detection Pipeline

This notebook implements an end-to-end object detection pipeline for identifying damaged car parts from photographs, using YOLOv8. It is intended to be run sequentially in **Google Colab** with a GPU runtime (`Runtime -> Change runtime type -> T4 GPU`).

## Motivation

The claims app (`src/lib/analyse.ts`) currently sends damage photos to OpenAI's or Anthropic's vision API and parses the returned text description. This notebook instead trains a dedicated object-detection model that identifies damaged car parts directly from an image and localizes them with bounding boxes, providing an in-house alternative to the external API call.

## Pipeline overview

1. **Problem framing** — defining the prediction target and output format (below)
2. **Data acquisition** — a dataset of car photos with damage already annotated
3. **EDA** — class balance, bounding-box size distribution, and visual inspection of a sample of labeled images before training
4. **Data preparation** — converting the dataset into the folder/label format required by YOLO
5. **Training** — fine-tuning a pretrained YOLOv8 model on the dataset
6. **Evaluation** — mAP, precision/recall, confusion matrix, and qualitative review of predictions
7. **Inference** — running the trained model on new images
8. **Backend integration** — mapping a YOLO prediction to the JSON structure expected by `src/lib/analyse.ts`

## Problem framing

Input: a single photograph of a damaged vehicle. Output: a list of detections, each consisting of a bounding box, a part name, and a confidence score (e.g., damage located near the bumper with 87% confidence).

This is framed as an object detection task rather than classification. A classifier would only indicate that a photo contains damage, without localization. Detection additionally provides location, which is required to highlight the corresponding zone on the 2D car diagram described in `docs/frontend-prd.md` §6.1.


## 1. Setup

The `ultralytics` package provides the YOLOv8 implementation, including the training loop, loss functions, augmentation, evaluation, and inference, behind a unified API. The `roboflow` package is used to download the dataset. The remaining dependencies are standard data-analysis and visualization libraries.


In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("No GPU — go to Runtime > Change runtime type > T4 GPU, then re-run this cell.")



## 2. Data acquisition

This notebook uses the [Car Damage Detection dataset by CAPSTONE](https://universe.roboflow.com/capstone-nh0nc/car-damage-detection-t0g92), published on Roboflow Universe:

- 3,226 images, CC BY 4.0 license (attribution required, otherwise free to use)
- 7 classes, corresponding to **car parts** rather than damage types: `Door, Light, Bonnet, Bumper, Dickey, Fender, Windshield` (Bonnet = hood, Dickey = boot/trunk, reflecting the Indian English part naming used by the original annotators)
- Provided in COCO format on the source site; the Roboflow SDK exports it directly in YOLOv8 format

**Note on dataset selection:** the CarDD dataset (which labels damage type and severity: dent, scratch, crack, glass shatter, flat tire, broken lamp) was initially considered. However, its public Hugging Face mirror includes images only, without annotation files, and the annotated version requires a manual licensing request to the original authors. The Roboflow dataset above was used instead; its classes indicate which part is affected rather than damage type or severity, an issue addressed separately in Section 8.

### Roboflow API key

An API key is required to download the dataset:

1. Create a free account at [app.roboflow.com](https://app.roboflow.com)
2. Go to Settings -> Roboflow API and copy the private API key
3. Paste it in the cell below


In [ ]:
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"  # <-- paste your key here

assert ROBOFLOW_API_KEY, "Set ROBOFLOW_API_KEY above before continuing."

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("capstone-nh0nc").project("car-damage-detection-t0g92")

# The project has 4 dataset versions as of when this notebook was written.
# If version(4) 404s, open the project page -> "Versions" tab and use whatever the latest is.
VERSION = 4
dataset = project.version(VERSION).download("yolov8")

print("Downloaded to:", dataset.location)



## 3. Exploratory data analysis

Prior to training, the dataset is examined along several dimensions: image counts per split, instance counts per class (class imbalance is a common cause of degraded per-class performance), bounding-box size distribution (small boxes are generally harder to detect than large ones), and a visual check that labels are correctly aligned with the corresponding image regions. This step helps identify labeling errors or data imbalance before they can be mistaken for model issues during training.


In [ ]:
import yaml
from pathlib import Path

data_yaml_path = Path(dataset.location) / "data.yaml"
with open(data_yaml_path) as f:
    data_cfg = yaml.safe_load(f)

print("Classes:", data_cfg["names"])
print("Number of classes:", data_cfg["nc"])

for split in ["train", "val", "test"]:
    folder_name = split if (Path(dataset.location) / split).exists() else ("valid" if split == "val" else split)
    img_dir = Path(dataset.location) / folder_name / "images"
    if img_dir.exists():
        n = len(list(img_dir.glob("*.*")))
        print(f"{split}: {n} images")



In [ ]:
# Class balance: count how many labeled instances of each class exist across the training split.
# A YOLO label file has one line per object: "class_id x_center y_center width height" (all
# normalized 0-1 relative to image size). We only need the class_id column here.

import pandas as pd
from collections import Counter

train_labels_dir = Path(dataset.location) / "train" / "labels"
class_names = data_cfg["names"]

counts = Counter()
box_sizes = []  # (width, height) normalized, for the size-distribution plot below

for label_file in train_labels_dir.glob("*.txt"):
    for line in label_file.read_text().strip().splitlines():
        if not line:
            continue
        parts = line.split()
        class_id = int(parts[0])
        w, h = float(parts[3]), float(parts[4])
        counts[class_names[class_id]] += 1
        box_sizes.append((w, h))

class_counts = pd.Series(counts).sort_values(ascending=False)
print(class_counts)



In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

class_counts.plot(kind="bar", ax=axes[0], color="tab:orange")
axes[0].set_title("Instances per class (training set)")
axes[0].set_ylabel("count")

widths = [w for w, h in box_sizes]
heights = [h for w, h in box_sizes]
axes[1].scatter(widths, heights, alpha=0.3, s=8)
axes[1].set_title("Box size distribution (normalized w vs h)")
axes[1].set_xlabel("width / image width")
axes[1].set_ylabel("height / image height")

plt.tight_layout()
plt.show()

# What to look for:
# - A class with far fewer instances than the rest will be predicted worse — you'd fix this later
#   with class-weighted loss, oversampling, or just gathering more examples of that class.
# - Boxes clustered near the bottom-left of the scatter are small relative to the image — YOLO
#   generally struggles more with small objects, which matters for something like a small scratch
#   vs a whole bumper.



In [ ]:
# Look at the labels drawn on actual images — the single most useful sanity check there is.
# If a box is clearly on the wrong part of the car, that's a labeling error you'd want to know
# about before training, not after wondering why the model is confused.

import cv2
import random

train_images_dir = Path(dataset.location) / "train" / "images"
sample_images = random.sample(list(train_images_dir.glob("*.*")), 6)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, sample_images):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    label_path = train_labels_dir / (img_path.stem + ".txt")
    if label_path.exists():
        for line in label_path.read_text().strip().splitlines():
            cls_id, xc, yc, bw, bh = line.split()[:5]
            cls_id = int(cls_id)
            xc, yc, bw, bh = float(xc) * w, float(yc) * h, float(bw) * w, float(bh) * h
            x1, y1 = int(xc - bw / 2), int(yc - bh / 2)
            x2, y2 = int(xc + bw / 2), int(yc + bh / 2)
            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 3)
            cv2.putText(img, class_names[cls_id], (x1, max(y1 - 8, 0)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
    ax.imshow(img)
    ax.axis("off")
plt.tight_layout()
plt.show()



## 4. Label format

Each image has a corresponding `.txt` file with one line per annotated part, in the format `class_id x_center y_center width height`, for example:

```
3 0.512 0.431 0.220 0.185
```

The final four values are normalized to the range 0-1 relative to image width/height, making the format resolution-independent. The accompanying `data.yaml` file specifies the `train`/`val` folder locations and the class_id-to-name mapping.


In [ ]:
print((Path(dataset.location) / "data.yaml").read_text())



## 5. Training

### Model architecture

YOLOv8 consists of a **backbone** (multi-scale feature extraction), a **neck** (feature fusion across scales, enabling detection of both large and small objects), and a **head**, which predicts bounding boxes and class probabilities directly per grid cell (anchor-free, unlike earlier YOLO versions). Training jointly optimizes bounding-box regression (CIoU loss), objectness, and classification loss in a single backward pass.

The `yolov8n.pt` checkpoint ("nano") is the smallest pretrained YOLOv8 variant, pretrained on COCO. It is fine-tuned here on the 7 target classes rather than trained from randomly initialized weights, following standard transfer-learning practice.


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # swap to yolov8s.pt for a bigger/slower/more-accurate model later

results = model.train(
    data=str(data_yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,       # stop early if val performance plateaus for 10 epochs
    project="car-damage-runs",
    name="yolov8n-run1",
)



## 6. Evaluation

`mAP@50` denotes mean average precision at an IoU threshold of 0.5 (a prediction is counted as correct if its box overlaps the ground truth by at least 50%). `mAP@50-95` averages this metric across a range of stricter IoU thresholds, providing a more comprehensive measure. As a reference point, Roboflow's own baseline model on this dataset reports **21.2% mAP@50**.


In [ ]:
metrics = model.val()
print("mAP@50:", metrics.box.map50)
print("mAP@50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)



In [ ]:
# Ultralytics saves a training-curve plot and a confusion matrix automatically. Display them.
from IPython.display import Image, display

run_dir = Path(results.save_dir)
display(Image(filename=str(run_dir / "results.png")))
display(Image(filename=str(run_dir / "confusion_matrix.png")))



## 7. Inference

The trained model is first evaluated qualitatively on a small set of validation images, allowing predictions to be compared against known ground-truth boxes, and then applied to a newly uploaded photo, representative of the images a customer would submit through the application's `/call` upload flow.


In [ ]:
val_images = list((Path(dataset.location) / "valid" / "images").glob("*.*"))[:4]
pred_results = model.predict(source=val_images, conf=0.25)

for r in pred_results:
    display(Image(r.plot()[:, :, ::-1]))  # BGR -> RGB for display; plot() draws boxes for you



In [ ]:
# Run inference on your own photo. Originally this used google.colab files.upload();
# locally, just set LOCAL_PHOTO to a path on disk. If unset, falls back to a validation image.
from pathlib import Path as _P

LOCAL_PHOTO = ""  # e.g. "/home/ottodev/Pictures/scratch-door.jpg"

uploaded_path = LOCAL_PHOTO if LOCAL_PHOTO and _P(LOCAL_PHOTO).exists() else str(val_images[0])
print("Using photo:", uploaded_path)

result = model.predict(source=uploaded_path, conf=0.25)[0]
display(Image(result.plot()[:, :, ::-1]))

for box in result.boxes:
    cls_name = class_names[int(box.cls)]
    conf = float(box.conf)
    print(f"{cls_name}: {conf:.2f}")



## 8. Backend integration

This section connects the trained model's output to the existing application. Currently, `analyseDamage()` in `src/lib/analyse.ts` sends the photo to OpenAI or Anthropic and parses the returned text via `findingsFromVisionText()` into `DamageFinding[]` objects with the following shape:

```ts
type DamageFinding = {
  area: string;
  observation: string;
  severity: "minor" | "moderate" | "severe" | "unknown";
  source: "vision" | "heuristic" | "customer";
};
```

The trained model can produce the same output structure directly. The cell below converts a YOLO prediction into this format, corresponding to the JSON payload the Next.js application would receive if this model were substituted for the OpenAI/Anthropic call.

Two limitations of the current approach:

1. Severity is not a label present in this dataset; the classes indicate which part is affected, not the extent of damage. Severity below is derived from a heuristic based on bounding-box area relative to the image, used to complete the output shape. A production version would require dedicated severity labels or a separate model.
2. Left/right and front/rear distinctions are not captured by the class names (e.g., "Bumper" or "Door" without side information). The mapping below assigns the more general diagram zone from `docs/frontend-prd.md` §6.1 rather than inferring an unsupported side, consistent with that document's guidance against placing findings on zones unsupported by the data.


In [ ]:
# Maps this dataset's part-name classes onto the car-diagram zones from frontend-prd.md §6.1.
# Bumper/Door/Light are genuinely ambiguous (front-or-rear, left-or-right) with this label set —
# left as the closest honest zone rather than guessed.
PART_TO_ZONE = {
    "Bonnet": "front",
    "Windshield": "windscreen",
    "Dickey": "rear",
    "Fender": "side (unspecified)",
    "Door": "side (unspecified)",
    "Bumper": "front-or-rear (unspecified)",
    "Light": "headlight (unspecified side)",
}

def severity_from_box_area(box, img_width, img_height):
    """Crude placeholder: bigger detected damage region -> treated as more severe.
    Not a real severity model -- see the note above. Replace this once you have real severity
    labels or a second model."""
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    box_area_frac = ((x2 - x1) * (y2 - y1)) / (img_width * img_height)
    if box_area_frac > 0.15:
        return "severe"
    if box_area_frac > 0.05:
        return "moderate"
    return "minor"

def yolo_result_to_damage_findings(result):
    """Converts one ultralytics Results object into the DamageFinding[] shape
    src/lib/analyse.ts and src/lib/types.ts already expect."""
    img_h, img_w = result.orig_shape
    findings = []
    for box in result.boxes:
        part = class_names[int(box.cls)]
        conf = float(box.conf)
        findings.append({
            "area": PART_TO_ZONE.get(part, part.lower()),
            "observation": f"Detected {part.lower()} damage, confidence {conf:.2f}.",
            "severity": severity_from_box_area(box, img_w, img_h),
            "source": "local_model",
        })
    return findings

import json
print(json.dumps(yolo_result_to_damage_findings(result), indent=2))



### Serve for Claimaroo (`LOCAL_VISION_URL`)

Run this cell **after** training finishes (needs `best.pt`). It starts FastAPI + a public Cloudflare tunnel and prints the URL to paste into Vercel / `.env.local` as `LOCAL_VISION_URL`.

Contract matches `src/lib/analyse.ts` `localModelAnalyse`:
`POST /analyse` with `{filename, mime_type, image_base64}` → `{observations, findings, confidence}` with `source: "local_model"`.

**Note:** Colab idle disconnects kill the tunnel. Start this cell shortly before the demo. The trycloudflare URL changes every rerun — update `LOCAL_VISION_URL` after each restart.



In [ ]:
# =============================================================================
# Claimaroo Vehicle Vision — serving cell (paste into the Colab notebook AFTER
# training finishes). Serves best.pt behind FastAPI and prints a public
# LOCAL_VISION_URL via a cloudflared tunnel (no account needed).
#
# Contract (matches src/lib/analyse.ts localModelAnalyse):
#   POST /analyse  {"filename", "mime_type", "image_base64"}
#   -> {"observations": [str], "findings": [{area, observation, severity, source}]}
# =============================================================================

!pip install -q ultralytics fastapi uvicorn nest_asyncio
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared

import base64, io, json, re, subprocess, threading, time

import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from PIL import Image
from ultralytics import YOLO

MODEL_PATH = "runs/detect/train/weights/best.pt"  # adjust if your run folder differs

# Honest mapping: this dataset labels car PARTS, not damage types, and
# Bumper/Door/Light/Fender are ambiguous without side info. Claimaroo's
# mapZone() needs left/right qualifiers to light the diagram, so those
# zones show as text findings instead of guessed map zones.
PART_TO_ZONE = {
    "Bonnet": "front",
    "Windshield": "windscreen",
    "Dickey": "rear",
    "Fender": "side (unspecified)",
    "Door": "side (unspecified)",
    "Bumper": "front-or-rear (unspecified)",
    "Light": "headlight (unspecified side)",
}

model = YOLO(MODEL_PATH)
class_names = model.names

# Stock COCO model, used only as a "does this show a vehicle?" gate. The
# damage model hallucinates parts on people/rooms; this keeps non-vehicle
# uploads from producing confident nonsense (they fall through to cloud
# vision instead, which describes what's actually in the image).
vehicle_model = YOLO("yolov8n.pt")
VEHICLE_COCO_IDS = {1, 2, 3, 5, 7}  # bicycle, car, motorcycle, bus, truck
MIN_DETECTION_CONF = 0.30


def vehicle_present(pil_image):
    for result in vehicle_model.predict(pil_image, verbose=False):
        for box in result.boxes:
            if int(box.cls) in VEHICLE_COCO_IDS and float(box.conf) >= 0.30:
                return True
    return False


def analysis_confidence(findings):
    if not findings:
        return "low"
    best = max(f["det_conf"] for f in findings)
    if best >= 0.65:
        return "high"
    if best >= 0.45:
        return "medium"
    return "low"


def severity_from_box_area(box, img_w, img_h):
    """Box-area heuristic placeholder — dataset has no severity labels."""
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    frac = ((x2 - x1) * (y2 - y1)) / (img_w * img_h)
    if frac > 0.15:
        return "severe"
    if frac > 0.05:
        return "moderate"
    return "minor"


app = FastAPI(title="Claimaroo Vehicle Vision")


@app.get("/health")
def health():
    return {"ok": True, "model": MODEL_PATH}


@app.post("/analyse")
def analyse(payload: dict):
    try:
        image = Image.open(io.BytesIO(base64.b64decode(payload.get("image_base64", ""))))
        image.load()
    except Exception:
        raise HTTPException(status_code=400, detail="image_base64 is not a decodable image")

    findings = []
    best_conf = 0.0
    for result in model.predict(image, verbose=False):
        img_h, img_w = result.orig_shape
        for box in result.boxes:
            conf = float(box.conf)
            best_conf = max(best_conf, conf)
            if conf < MIN_DETECTION_CONF:
                continue
            part = class_names[int(box.cls)]
            findings.append({
                "area": PART_TO_ZONE.get(part, part.lower()),
                "observation": f"Detected {part.lower()} damage, confidence {conf:.2f}.",
                "severity": severity_from_box_area(box, img_w, img_h),
                "source": "local_model",
                "det_conf": conf,
            })

    # Gate: accept only if a vehicle is visible OR a part detection is strong
    # enough to imply a car (close-ups of damage often show no full vehicle —
    # calibrated on a selfie whose strongest hallucinated box scored 0.26).
    if not (vehicle_present(image) or best_conf >= 0.35):
        return {"observations": [], "findings": []}

    confidence = analysis_confidence(findings)

    if not findings:
        findings = [{
            "area": "observed",
            "observation": "Vehicle present but no damage parts detected by the local model.",
            "severity": "unknown",
            "source": "local_model",
        }]
        confidence = "low"
        return {"observations": [findings[0]["observation"]],
                "findings": [{k: v for k, v in findings[0].items() if k != "det_conf"}],
                "confidence": confidence}

    return {"observations": [f["observation"] for f in findings],
            "findings": [{k: v for k, v in f.items() if k != "det_conf"} for f in findings],
            "confidence": confidence}


nest_asyncio.apply()
threading.Thread(
    target=uvicorn.run, args=(app,), kwargs={"host": "127.0.0.1", "port": 8000, "log_level": "warning"},
    daemon=True,
).start()
time.sleep(3)

tunnel = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url = None
deadline = time.time() + 60
while time.time() < deadline and url is None:
    line = tunnel.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if m:
        url = m.group(0)
if not url:
    raise RuntimeError("cloudflared did not produce a URL — rerun this cell")

import requests
health = requests.get(f"{url}/health", timeout=30).json()
print("tunnel:", url)
print("health:", health)
print()
print("Set this in Claimaroo (.env.local locally, LOCAL_VISION_URL in Vercel):")
print(f'LOCAL_VISION_URL={url}/analyse')
print()
print("NOTE: the trycloudflare URL changes every time you rerun this cell —")
print("update LOCAL_VISION_URL after each restart. Colab also disconnects idle")
print("runtimes, so start the tunnel shortly before your demo.")



## 9. Summary and next steps

This notebook followed the standard supervised-learning workflow — problem framing, data acquisition, exploratory data analysis, data format review, training, evaluation, inference, and integration planning — applied to an object detection task.

**Potential next steps, in approximate order of effort:**

1. Extend training or switch to `yolov8s.pt` if mAP@50 is insufficient (larger model, slower, generally more accurate)
2. Analyze misclassifications using the confusion matrix and manual review of low-confidence predictions before increasing training duration
3. Resolve front/rear and left/right ambiguity, either through a finer-grained labeled dataset or a positional heuristic based on bounding-box location within the image
4. Obtain severity labels, either through manual annotation of a subset of the data or a dedicated severity model
5. Once model accuracy is satisfactory, implement the FastAPI service and integrate it into `analyse.ts` as outlined in Section 8
